# The Clipped Group-Relative Objective

Implement old-policy ratios, signed clipping, group advantages, masks, and reference KL.

## Before You Start

Course 12. Entry course: 10. Budget about 75 minutes for reading, code, and one CPU experiment; optional sweeps take longer.

**Concept prerequisites:** rl-03_reinforce.

## What You Will Be Able to Do

- Compute the full component objective
- Separate old policy from reference policy
- Compare actual vanilla and clipped updates

## Three policies have different jobs

The current policy is optimized. The old rollout policy generated the sampled completions and is fixed during their update epochs. A separate frozen reference policy anchors drift, often to an SFT checkpoint. The importance ratio is exp(logp_current-logp_old), not exp(logp_current-logp_reference). Refreshing old probabilities inside each update removes the intended trust-region comparison.

For rewards R_i within one prompt group, A_i=(R_i-mean(R))/(population_std(R)+1e-8). With outcome rewards this scalar is shared across valid response tokens. The surrogate is min(ratio*A, clip(ratio,1-epsilon,1+epsilon)*A). The minimum must be taken after multiplying by signed advantages: negative advantages reverse which ratio direction is harmful.

## Normalize and regularize explicitly

The lab follows the original per-response then per-group averaging: mean_i mean_valid_tokens [surrogate-beta*KL_sample]. Loss is its negative. Variable response lengths therefore receive equal response-level weight. Padding and prompt tokens must not enter these means. Reject a response with no valid target tokens.

The nonnegative sampled KL term is exp(logp_reference-logp_current)-(logp_reference-logp_current)-1. Under samples from the current policy its expectation estimates KL(current||reference); with old-policy rollouts it is used within a surrogate update, not an exact on-policy KL measurement at arbitrary drift. Equal-reward groups have zero task advantages, although KL can still act. The notebook performs actual updates on a tiny categorical task, comparing vanilla REINFORCE and clipped group updates at a disclosed sample budget. Neither method is a universal winner.

## Worked Example: Clipping depends on advantage sign

ratio=1.4 and epsilon=.2.

1. **Positive advantage:** With A=2, min(2.8,2.4)=2.4: further increasing a rewarded action is clipped.
2. **Negative advantage:** With A=-2, min(-2.8,-2.4)=-2.8: the harmful increase remains penalized.
3. **Different reference:** A reference-policy probability affects KL, not this old-policy ratio.

Clipping the final loss or taking an absolute advantage implements a different algorithm.

## Coding Contract

Implement the public functions below. Private helpers are supplied where the exercise focuses on composition. NumPy/pure Python runs in the browser; the separate experiment uses autograd or a simulator when needed.

```python
# Generated starter; edit the functions in the course editor or notebook.
"""One prompt group, original response-weighted clipped GRPO surrogate."""
import numpy as np

def clipped_grpo(logp, old_logp, reference_logp, rewards, mask, epsilon=0.2, beta=0.01):
    """Return dict loss, advantages, ratios, kl, clip_fraction.

Log arrays finite [G,T], rewards finite [G], Boolean mask [G,T]. G>=2;
each response has >=1 valid token. 0<epsilon<1, beta>=0; invalid: ValueError.
Population std with +1e-8. Loss=-mean_G(mean_valid(min(r*A,clip(r)*A)-beta*KL)).
KL=exp(reference_logp-logp)-(reference_logp-logp)-1. Padded entries in
ratios/kl are zero, and must not be exponentiated. clip_fraction is the
fraction of valid ratios outside [1-epsilon,1+epsilon]. No input mutation."""
    raise NotImplementedError('Implement this function, then run the checks.')
```

## Check Your Understanding

Rewards=[1,3]. Use population standard deviation and ignore tiny epsilon for this calculation.

- Advantage of reward 3

<details><summary>Worked answers</summary>

- 1: Mean=2, std=1, so advantage=(3-2)/1=1.

</details>

## Experiment

Run `experiments/rl_group_updates.py` from the repository, or use the self-contained notebook linked below the editor. Start with the declared CPU defaults. The notebook includes any required local model source; no model weights, datasets, API keys, or paid GPU are needed.

- Report sampled trajectories, optimizer steps, reward, KL, clipping fraction, and seed for each method.
- The browser challenge verifies the objective; the notebook verifies gradients and real policy changes. It is not a language-model benchmark reproduction.

## Transfer Problems

1. Compare response-weighted and token-weighted objectives on unequal lengths.
2. Freeze an incorrect old policy and predict how the ratio test will fail.

For each transfer, submit the changed code, one predicted outcome, actual measurements, one negative control, and a limitation. These open experiments need a defensible result, not a predetermined winning curve.

## AI-Assisted Coding Route

1. Predict the worked example and one edge case without assistance.
2. Ask an assistant for one small function or a test critique, not the completed notebook. Supply the exact shapes and contract.
3. Explain every returned operation and inspect the diff. Reject changes to the grader or the intended objective.
4. Run the public tests, then deliberately break a mechanism named in this lesson. Confirm that a behavioral test fails.
5. Compare with the worked solution only after an attempt. Close it and reimplement the difficult part from memory.
6. Record what you could explain unaided, what the assistant supplied, and which measured result supports your conclusion.

## Completion Evidence

A passing grader establishes this small function contract, not research reproduction or production readiness. Submit a reproducible experiment report with seed, split, shapes, budget, baseline, held-out result, failure case, and limitation. Keep failed seeds. Answers to historical/discussion questions should distinguish fact, inference, and an open empirical claim.

## Primary Source

[DeepSeekMath: GRPO](https://arxiv.org/html/2402.03300v3#S4.SS1.SSS1). The numerical examples, code, and synthetic datasets here are original teaching material; the source is for the underlying mechanism and its research context.



## Runtime

CPU is sufficient. Install the libraries below in a fresh environment. Run the starter and checks after implementing the functions; an untouched starter is expected to fail. Run the reference only when ready to compare. Training experiments are mechanism-scale, not paper benchmark reproductions.


In [ ]:
%pip install -q "numpy>=1.26,<3" "torch>=2.2,<3"


In [ ]:
# Generated starter; edit the functions in the course editor or notebook.
"""One prompt group, original response-weighted clipped GRPO surrogate."""
import numpy as np

def clipped_grpo(logp, old_logp, reference_logp, rewards, mask, epsilon=0.2, beta=0.01):
    """Return dict loss, advantages, ratios, kl, clip_fraction.

Log arrays finite [G,T], rewards finite [G], Boolean mask [G,T]. G>=2;
each response has >=1 valid token. 0<epsilon<1, beta>=0; invalid: ValueError.
Population std with +1e-8. Loss=-mean_G(mean_valid(min(r*A,clip(r)*A)-beta*KL)).
KL=exp(reference_logp-logp)-(reference_logp-logp)-1. Padded entries in
ratios/kl are zero, and must not be exponentiated. clip_fraction is the
fraction of valid ratios outside [1-epsilon,1+epsilon]. No input mutation."""
    raise NotImplementedError('Implement this function, then run the checks.')


In [ ]:
import numpy as np
old = np.log(np.full((3, 3), .25))
ratios = np.array([[1.4,.7,1.], [1.4,.7,1.1], [1.1,.8,1.2]])
z = old+np.log(ratios); ref = np.log(np.full((3,3), .2))
r = np.array([0.,2.,5.]); m = np.array([[1,0,0],[1,1,1],[1,1,0]], bool)
adv = (r-r.mean())/(r.std()+1e-8)
per_response = []
for i in range(3):
    values = []
    for t in np.flatnonzero(m[i]):
        ratio = np.exp(z[i,t]-old[i,t]); delta = ref[i,t]-z[i,t]
        values.append(min(ratio*adv[i], max(.8,min(1.2,ratio))*adv[i])-.3*(np.exp(delta)-delta-1))
    per_response.append(np.mean(values))
got = clipped_grpo(z,old,ref,r,m,beta=.3)
assert np.isclose(got['loss'], -np.mean(per_response)), 'Signed clipping, reference KL, and response weighting'
assert np.allclose(got['advantages'], adv)
assert np.allclose(got['ratios'][m], ratios[m]) and np.all(got['ratios'][~m] == 0)
assert np.isclose(got['clip_fraction'], .5)
assert abs(clipped_grpo(old,old,old,np.ones(3),m)['loss']) < 1e-12
z2 = z.copy(); z2[~m] = 10000
assert np.isclose(clipped_grpo(z2,old,ref,r,m,beta=.3)['loss'], got['loss']), 'Never exponentiate padding'
try:
    clipped_grpo(z,old,ref,r,np.zeros_like(m))
except ValueError:
    pass
else:
    raise AssertionError('Reject empty responses')
print('Full clipped GRPO checks passed.')


## Worked Solution

Read the explanation and docstrings alongside the code. Verify nonmutation, masks, normalization, and numerical conventions against your own implementation.


In [ ]:
"""One prompt group, original response-weighted clipped GRPO surrogate."""
import numpy as np


def clipped_grpo(logp, old_logp, reference_logp, rewards, mask, epsilon=.2, beta=.01):
    """Return dict loss, advantages, ratios, kl, clip_fraction.

    Log arrays finite [G,T], rewards finite [G], Boolean mask [G,T]. G>=2;
    each response has >=1 valid token. 0<epsilon<1, beta>=0; invalid: ValueError.
    Population std with +1e-8. Loss=-mean_G(mean_valid(min(r*A,clip(r)*A)-beta*KL)).
    KL=exp(reference_logp-logp)-(reference_logp-logp)-1. Padded entries in
    ratios/kl are zero, and must not be exponentiated. clip_fraction is the
    fraction of valid ratios outside [1-epsilon,1+epsilon]. No input mutation.
    """
    z, old, ref = [np.asarray(v, float) for v in (logp, old_logp, reference_logp)]
    rewards, m = np.asarray(rewards, float), np.asarray(mask)
    if z.ndim != 2 or old.shape != z.shape or ref.shape != z.shape or m.shape != z.shape or rewards.shape != (z.shape[0],) or z.shape[0] < 2 or m.dtype.kind != 'b' or not m.any(axis=1).all():
        raise ValueError('invalid group shapes or masks')
    if not 0 < epsilon < 1 or not np.isfinite(beta) or beta < 0 or not all(np.isfinite(v).all() for v in (z, old, ref, rewards)):
        raise ValueError('invalid objective inputs')
    advantages = (rewards-rewards.mean())/(rewards.std()+1e-8)
    ratios, kl = np.zeros_like(z), np.zeros_like(z)
    ratios[m] = np.exp(z[m]-old[m])
    delta = ref[m]-z[m]
    kl[m] = np.expm1(delta)-delta
    a = advantages[:, None]
    objective = np.minimum(ratios*a, np.clip(ratios,1-epsilon,1+epsilon)*a)-beta*kl
    loss = -np.mean(np.where(m, objective, 0).sum(-1)/m.sum(-1))
    fraction = np.mean((ratios[m] < 1-epsilon) | (ratios[m] > 1+epsilon))
    return dict(loss=float(loss), advantages=advantages, ratios=ratios, kl=kl, clip_fraction=float(fraction))


In [ ]:
import numpy as np
old = np.log(np.full((3, 3), .25))
ratios = np.array([[1.4,.7,1.], [1.4,.7,1.1], [1.1,.8,1.2]])
z = old+np.log(ratios); ref = np.log(np.full((3,3), .2))
r = np.array([0.,2.,5.]); m = np.array([[1,0,0],[1,1,1],[1,1,0]], bool)
adv = (r-r.mean())/(r.std()+1e-8)
per_response = []
for i in range(3):
    values = []
    for t in np.flatnonzero(m[i]):
        ratio = np.exp(z[i,t]-old[i,t]); delta = ref[i,t]-z[i,t]
        values.append(min(ratio*adv[i], max(.8,min(1.2,ratio))*adv[i])-.3*(np.exp(delta)-delta-1))
    per_response.append(np.mean(values))
got = clipped_grpo(z,old,ref,r,m,beta=.3)
assert np.isclose(got['loss'], -np.mean(per_response)), 'Signed clipping, reference KL, and response weighting'
assert np.allclose(got['advantages'], adv)
assert np.allclose(got['ratios'][m], ratios[m]) and np.all(got['ratios'][~m] == 0)
assert np.isclose(got['clip_fraction'], .5)
assert abs(clipped_grpo(old,old,old,np.ones(3),m)['loss']) < 1e-12
z2 = z.copy(); z2[~m] = 10000
assert np.isclose(clipped_grpo(z2,old,ref,r,m,beta=.3)['loss'], got['loss']), 'Never exponentiate padding'
try:
    clipped_grpo(z,old,ref,r,np.zeros_like(m))
except ValueError:
    pass
else:
    raise AssertionError('Reject empty responses')
print('Full clipped GRPO checks passed.')


## Measured Experiment

Predict before running. Change one variable at a time. Report actual outcomes and failures; the lesson does not promise a successful learning curve for every configuration.


In [ ]:
"""Actual vanilla/GRPO categorical updates with equal rollout and update budgets.

One-step completions isolate the objective. This is not an LLM benchmark.
"""
import json
import torch


def group_experiment():
    torch.set_num_threads(1)
    reports = []
    true_rewards = torch.tensor([0., .2, .5, 1.])
    for seed in [3, 9, 17]:
        for method in ['vanilla', 'grpo', 'grpo_bad_reward']:
            torch.manual_seed(seed)
            logits = torch.nn.Parameter(torch.zeros(4))
            optimizer = torch.optim.Adam([logits], lr=.05)
            reference = torch.full((4,), .25).log()
            curve = []
            observed_rewards = true_rewards if method != 'grpo_bad_reward' else torch.tensor([1., .2, .5, .1])
            rounds, group_size, epochs = (100, 16, 1) if method == 'vanilla' else (50, 32, 2)
            for step in range(rounds):
                old = logits.detach().log_softmax(-1)
                actions = torch.multinomial(old.exp(), group_size, replacement=True)
                rewards = observed_rewards[actions]
                advantages = (rewards-rewards.mean())/(rewards.std(unbiased=False)+1e-8)
                for _ in range(epochs):
                    current = logits.log_softmax(-1)
                    chosen = current[actions]
                    ratio = (chosen-old[actions]).exp()
                    delta = reference[actions]-chosen
                    kl = delta.exp()-delta-1
                    if method == 'vanilla':
                        # Exact expected reward under the old policy is an
                        # action-independent baseline in this enumerable toy.
                        baseline = (old.exp()*observed_rewards).sum()
                        loss = -(chosen*(rewards-baseline)).mean()
                    else:
                        loss = -(torch.minimum(ratio*advantages,ratio.clamp(.8,1.2)*advantages)-.02*kl).mean()
                    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
                with torch.no_grad():
                    p = logits.softmax(-1)
                    curve.append(dict(step=step+1, proxy_reward=float(p@observed_rewards), true_reward=float(p@true_rewards), exact_kl=float((p*(p.log()-reference)).sum()), clip_fraction=float(((ratio<.8)|(ratio>1.2)).float().mean())))
            assert not torch.equal(logits.detach(),torch.zeros(4))
            reports.append(dict(seed=seed,method=method,rollouts=1600,optimizer_steps=100,curve=curve,
                zero_variance_advantages=((torch.ones(4)-1)/(torch.zeros(())+1e-8)).tolist(),
                group_size=group_size,epochs_per_rollout=epochs,
                limitation='Matched total samples and optimizer steps, not FLOPs or group size. Vanilla uses fresh on-policy samples for every update; GRPO reuses each rollout for two epochs.'))
    return reports


if __name__ == '__main__':
    print(json.dumps(group_experiment(), indent=2))
